# Chapter 3 publication and verification

Run after the completed numerical notebook. This notebook creates release evidence and Earth Engine assets; browser candidate testing and live publication remain separate required steps.

In [ ]:
from pathlib import Path
import sys, json, hashlib, importlib.metadata as metadata
ROOT=Path('/content/drive/MyDrive/PHD/CSP3_GPP_outputs')
OUTPUT=ROOT/'CH3_profitability/20261007_profitability_v1'
CODE=OUTPUT/'publication_code'
for row in json.loads((OUTPUT/'publication_code_manifest.json').read_text()):
    assert hashlib.sha256((CODE/row['file']).read_bytes()).hexdigest()==row['sha256'],row['file']
assert json.loads((OUTPUT/'validation.json').read_text())['verified']
sys.path.insert(0,str(CODE))
from ch3_profitability.audit_release import run as audit
audit(OUTPUT)
print('Verified publication code and numerical prerequisites')

In [ ]:
import ee
from google.cloud import storage
versions={name:metadata.version(name) for name in ['earthengine-api','google-cloud-storage','numpy','pandas','rasterio','geopandas']}
(OUTPUT/'publication_environment.json').write_text(json.dumps(versions,indent=2))
try:
    ee.Initialize(project='ee-njberkowitz95')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='ee-njberkowitz95')
print('Earth Engine connected; dependency versions recorded')

In [ ]:
from ch3_profitability.assets import prepare
prepare(OUTPUT)
print('Earth Engine tables and temporal image prepared')

In [ ]:
from ch3_profitability.assets import submit
tasks=submit(OUTPUT)
print(json.dumps(tasks,indent=2))

In [ ]:
from ch3_profitability.supplement import run as figures
from ch3_profitability.report import run as report
from ch3_profitability.maps import run as maps
from ch3_profitability.identity import run as identities
figures(OUTPUT)
maps(OUTPUT)
report(OUTPUT)
identities(ROOT,OUTPUT)
print('Publication figures, report and preserved identities complete')

In [ ]:
import time
for attempt in range(60):
    ids=[x['response']['id'] for x in tasks if 'response' in x]
    states=ee.data.getTaskStatus(ids) if ids else []
    print([(x['state'],x.get('error_message')) for x in states],flush=True)
    if any(x['state'] in ['FAILED','CANCELLED'] for x in states): raise RuntimeError('An Earth Engine import failed')
    if all(x['state']=='COMPLETED' for x in states): break
    time.sleep(10)
else: raise RuntimeError('Imports still running; repeat this status cell later')

In [ ]:
from ch3_profitability.assets import verify
from ch3_profitability.verify_native import run as verify_native
verify(OUTPUT)
verify_native(ROOT)
print('New assets and reused native input assets verified')

In [ ]:
from ch4_marginality.profit_workspace import lock_inputs
from ch3_profitability.pipeline import SOURCE_RELEASE
from ch3_profitability.build_app import build
prior=ROOT/'CH4_marginality'/SOURCE_RELEASE
lock_inputs(ROOT,prior/'analysis',OUTPUT,json.loads((prior/'input_manifest.json').read_text()))
candidate=build(OUTPUT)
print('Verified app candidate:',candidate)
print('Browser testing and publication are still required; this notebook does not declare the live app complete.')